# 06 - Maintenance Schedule Output
**Vehicle Predictive Maintenance Project**

---

## Purpose of This Notebook

This is the final notebook in the project. It takes all outputs from the previous notebooks and produces a single, business-facing Excel workbook that a fleet manager can open and act on immediately.

### What the output contains
The Excel workbook has three sheets:

1. **Summary** — fleet-level overview: total vehicles, risk breakdown, how many flags of each type, generated date
2. **Maintenance Schedule** — one row per active vehicle, sorted by risk (High first), with risk band, specific flags, recommended action, and driver info
3. **High Risk Detail** — filtered view of High risk vehicles only, for easy printing or sharing with the maintenance team

### Recommended Action Logic
Each vehicle gets an auto-generated plain English recommendation based on its flags:
- Safety-critical flags (brakes/tyres) always appear first
- Electrical, Engine, Cooling flags follow
- MOT due within 90 days is noted
- Low risk vehicles with no flags get a 'No immediate action required' note

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side, GradientFill
)
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
import datetime
import warnings
warnings.filterwarnings('ignore')

PROCESSED = '../data/processed/'
OUTPUTS   = '../outputs/'

risk_scores      = pd.read_csv(PROCESSED + 'risk_scores.csv')
survival_preds = pd.read_csv(PROCESSED + 'survival_predictions.csv')
master           = pd.read_csv(PROCESSED + 'master_vehicles.csv', parse_dates=['Registered Date'])
features_active  = pd.read_csv(PROCESSED + 'features_active.csv')

GENERATED_DATE = datetime.date.today().strftime('%d %B %Y')

print('Data loaded.')
print(f'  Risk scores:       {len(risk_scores):,} vehicles')
print(f'  Generated date:    {GENERATED_DATE}')

## 2. Build the Master Output Table

### Why
We join all the relevant information together into a single clean table — one row per active vehicle — before writing to Excel. This keeps the Excel writing code simple and the data preparation transparent.

In [ ]:
# Start with risk scores as base
output = risk_scores[[
    'Registration', 'Risk_Score', 'Risk_Label_Predicted',
    'Brakes_Flag', 'Wheels_Tyres_Flag',
    'Vehicle Age (Years)', 'DriverScore', 'Days_Since_Last_Repair'
]].copy()

output = output.rename(columns={'Risk_Label_Predicted': 'Risk_Band'})

# Join vehicle info from master
vehicle_info = master[[
    'Registration', 'Make', 'Model', 'Asset Type', 'Vehicle Status'
]].drop_duplicates('Registration')
output = output.merge(vehicle_info, on='Registration', how='left')

# Join driver info and MOT from survival predictions
surv_cols = survival_preds[[
    'Registration',
    'Prob_Engine_90d', 'Prob_Electrical_90d', 'Prob_Cooling_90d',
    'Days_Since_Brakes', 'Days_Since_Wheels_Tyres'
]].copy() if 'Days_Since_Brakes' in survival_preds.columns else survival_preds[[
    'Registration', 'Prob_Engine_90d', 'Prob_Electrical_90d', 'Prob_Cooling_90d'
]].copy()
output = output.merge(surv_cols, on='Registration', how='left')

# Join driver name from features_active
if 'Driver' in features_active.columns:
    output = output.merge(
        features_active[['Registration', 'Driver', 'Branch', 'Next MOT Date', 'Days Until MOT']],
        on='Registration', how='left'
    )

# MOT flag — due within 90 days
if 'Days Until MOT' in output.columns:
    output['MOT_Due_90_Days'] = output['Days Until MOT'].between(0, 90).astype(int)
    output['MOT_Overdue']     = (output['Days Until MOT'] < 0).astype(int)

print(f'Output table built: {output.shape[0]} vehicles x {output.shape[1]} columns')
print(output.dtypes)

## 3. Generate Recommended Actions

### Why
Rather than asking the fleet manager to interpret flags manually, we auto-generate a plain English recommendation for each vehicle. Safety-critical items (brakes, tyres) are always mentioned first. The recommendation is proportionate to the risk band — High risk vehicles get specific action items, Low risk vehicles get a brief confirmation that no action is needed.

In [ ]:
def generate_recommendation(row):
    actions = []

    # Safety critical first
    if row.get('Brakes_Flag', 0) == 1:
        actions.append('Book brake inspection — no recorded brake service in last 24 months')
    if row.get('Wheels_Tyres_Flag', 0) == 1:
        actions.append('Check tyre condition — no recorded tyre service in last 24 months')

    # Survival model flags
    if row.get('Prob_Electrical_90d', 0) > 0.15:
        actions.append('Electrical system check recommended — elevated fault probability')
    if row.get('Prob_Engine_90d', 0) > 0.05:
        actions.append('Engine inspection recommended')
    if row.get('Prob_Cooling_90d', 0) > 0.05:
        actions.append('Cooling system check recommended')

    # MOT
    if row.get('MOT_Overdue', 0) == 1:
        actions.append('URGENT: MOT overdue — vehicle should not be on road')
    elif row.get('MOT_Due_90_Days', 0) == 1:
        days = int(row.get('Days Until MOT', 0))
        actions.append(f'MOT due in {days} days — book now')

    # Long gap since any repair
    days_since = row.get('Days_Since_Last_Repair', 0)
    if days_since not in [999, np.nan, None] and days_since > 365:
        actions.append(f'No recorded repair in over 12 months — general inspection advised')

    if not actions:
        if row.get('Risk_Band') == 'Low':
            return 'No immediate action required — continue routine monitoring'
        else:
            return 'Monitor — review at next scheduled service'

    return ' | '.join(actions)

output['Recommended Action'] = output.apply(generate_recommendation, axis=1)

print('Recommendations generated.')
print('\nSample recommendations by risk band:')
for band in ['High', 'Medium', 'Low']:
    sample = output[output['Risk_Band'] == band]['Recommended Action'].iloc[0]
    print(f'\n  {band}: {sample}')

## 4. Prepare Final Column Set

### Why
We select and rename columns to be clean and readable for a non-technical audience. Internal model columns (encoded features, log-transformed values etc.) are excluded.

In [ ]:
# Build clean display table
display_cols = {
    'Registration':        'Registration',
    'Make':                'Make',
    'Model':               'Model',
    'Asset Type':          'Van Type',
    'Driver':              'Driver',
    'Branch':              'Branch',
    'Vehicle Age (Years)': 'Vehicle Age (Yrs)',
    'DriverScore':         'Driver Score',
    'Risk_Band':           'Risk Band',
    'Risk_Score':          'Risk Score',
    'Brakes_Flag':         'Brakes Overdue',
    'Wheels_Tyres_Flag':   'Tyres Overdue',
    'Days_Since_Last_Repair': 'Days Since Last Repair',
    'Next MOT Date':       'Next MOT Date',
    'Days Until MOT':      'Days Until MOT',
    'Recommended Action':  'Recommended Action'
}

# Only keep columns that exist
available_cols = {k: v for k, v in display_cols.items() if k in output.columns}
schedule = output[list(available_cols.keys())].copy()
schedule = schedule.rename(columns=available_cols)

# Replace 999 Days Since Last Repair with 'No record'
if 'Days Since Last Repair' in schedule.columns:
    schedule['Days Since Last Repair'] = schedule['Days Since Last Repair'].apply(
        lambda x: 'No record' if x == 999 else (int(x) if pd.notna(x) else 'No record')
    )

# Replace flag 1/0 with Yes/No
for col in ['Brakes Overdue', 'Tyres Overdue']:
    if col in schedule.columns:
        schedule[col] = schedule[col].map({1: 'Yes', 0: 'No'})

# Sort High first, then by Risk Score descending
risk_order = {'High': 0, 'Medium': 1, 'Low': 2}
schedule['_sort'] = schedule['Risk Band'].map(risk_order)
schedule = schedule.sort_values(['_sort', 'Risk Score'], ascending=[True, False])
schedule = schedule.drop(columns=['_sort'])
schedule['Vehicle Age (Yrs)'] = schedule['Vehicle Age (Yrs)'].round(1)
schedule['Driver Score']      = schedule['Driver Score'].round(1)

print(f'Final schedule: {len(schedule)} vehicles x {len(schedule.columns)} columns')
display(schedule.head(5))

## 5. Build Excel Workbook

### Why
We use openpyxl to build a properly formatted, professional Excel workbook with three sheets. Colour coding follows a clear logic — red for High risk, amber for Medium, green for Low — so the fleet manager can scan the sheet quickly without reading every row.

In [ ]:
# Colour palette
COLOURS = {
    'High':        'FFE74C3C',  # Red
    'Medium':      'FFF39C12',  # Amber
    'Low':         'FF2ECC71',  # Green
    'header_bg':   'FF2C3E50',  # Dark navy
    'header_font': 'FFFFFFFF',  # White
    'alt_row':     'FFF2F3F4',  # Light grey
    'title_bg':    'FF1A252F',  # Darker navy
}

def header_style(cell, text):
    cell.value = text
    cell.font  = Font(bold=True, color=COLOURS['header_font'], name='Arial', size=10)
    cell.fill  = PatternFill('solid', start_color=COLOURS['header_bg'])
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def risk_fill(risk_band):
    return PatternFill('solid', start_color=COLOURS.get(risk_band, 'FFFFFFFF'))

def thin_border():
    side = Side(style='thin', color='FFD5D8DC')
    return Border(left=side, right=side, top=side, bottom=side)

print('Style helpers defined.')

In [ ]:
wb = Workbook()

# ── SHEET 1: SUMMARY ──────────────────────────────────────────────────────────
ws_summary = wb.active
ws_summary.title = 'Summary'
ws_summary.sheet_view.showGridLines = False

# Title
ws_summary.merge_cells('B2:F2')
title_cell = ws_summary['B2']
title_cell.value = 'Fleet Maintenance Risk Report'
title_cell.font  = Font(bold=True, size=18, name='Arial', color='FFFFFFFF')
title_cell.fill  = PatternFill('solid', start_color=COLOURS['title_bg'])
title_cell.alignment = Alignment(horizontal='center', vertical='center')
ws_summary.row_dimensions[2].height = 40

ws_summary.merge_cells('B3:F3')
sub_cell = ws_summary['B3']
sub_cell.value = f'Generated: {GENERATED_DATE}'
sub_cell.font  = Font(italic=True, size=10, name='Arial', color='FF7F8C8D')
sub_cell.alignment = Alignment(horizontal='center')

# Fleet overview stats
stats = [
    ('Total Active Vehicles',    len(schedule),                                  COLOURS['header_bg']),
    ('High Risk',                (schedule['Risk Band'] == 'High').sum(),         COLOURS['High']),
    ('Medium Risk',              (schedule['Risk Band'] == 'Medium').sum(),        COLOURS['Medium']),
    ('Low Risk',                 (schedule['Risk Band'] == 'Low').sum(),           COLOURS['Low']),
    ('Brakes Overdue',           (schedule.get('Brakes Overdue', pd.Series()) == 'Yes').sum(), 'FFEC407A'),
    ('Tyres Overdue',            (schedule.get('Tyres Overdue', pd.Series()) == 'Yes').sum(),  'FFAB47BC'),
]

row = 5
for label, value, colour in stats:
    # Label cell
    lc = ws_summary.cell(row=row, column=2, value=label)
    lc.font      = Font(bold=True, name='Arial', size=11)
    lc.alignment = Alignment(vertical='center')
    ws_summary.row_dimensions[row].height = 28

    # Value cell
    vc = ws_summary.cell(row=row, column=4, value=value)
    vc.font      = Font(bold=True, name='Arial', size=14, color='FFFFFFFF')
    vc.fill      = PatternFill('solid', start_color=colour)
    vc.alignment = Alignment(horizontal='center', vertical='center')
    vc.border    = thin_border()
    row += 1

# Notes section
row += 1
notes = [
    'NOTES:',
    f'• High Risk: vehicles scoring 5+ on composite risk index',
    f'• Brakes/Tyres Overdue: active vans over 4 years old with no recorded service in 24+ months',
    f'• Vehicles under 4 years old excluded from brakes/tyres flag (likely under dealer warranty)',
    f'• Driver scores sourced from telematics — missing scores replaced with fleet average',
    f'• Survival model covers Engine, Electrical, and Cooling categories only',
]
for note in notes:
    nc = ws_summary.cell(row=row, column=2, value=note)
    nc.font = Font(name='Arial', size=9, bold=(note == 'NOTES:'), color='FF555555')
    row += 1

# Column widths
ws_summary.column_dimensions['A'].width = 3
ws_summary.column_dimensions['B'].width = 35
ws_summary.column_dimensions['C'].width = 5
ws_summary.column_dimensions['D'].width = 15

print('Summary sheet built.')

In [ ]:
# ── SHEET 2: MAINTENANCE SCHEDULE ─────────────────────────────────────────────
ws_sched = wb.create_sheet('Maintenance Schedule')
ws_sched.sheet_view.showGridLines = False
ws_sched.freeze_panes = 'A2'  # Freeze header row

cols = list(schedule.columns)

# Header row
for col_idx, col_name in enumerate(cols, start=1):
    header_style(ws_sched.cell(row=1, column=col_idx), col_name)
ws_sched.row_dimensions[1].height = 30

# Data rows
risk_band_col = cols.index('Risk Band') + 1

for row_idx, (_, row_data) in enumerate(schedule.iterrows(), start=2):
    risk_band = row_data['Risk Band']
    is_alt    = (row_idx % 2 == 0)

    for col_idx, col_name in enumerate(cols, start=1):
        cell       = ws_sched.cell(row=row_idx, column=col_idx)
        cell.value = row_data[col_name]
        cell.font  = Font(name='Arial', size=9)
        cell.border = thin_border()
        cell.alignment = Alignment(vertical='center', wrap_text=(col_name == 'Recommended Action'))

        # Colour the Risk Band cell
        if col_name == 'Risk Band':
            cell.fill = risk_fill(risk_band)
            cell.font = Font(bold=True, name='Arial', size=9,
                             color='FFFFFFFF' if risk_band == 'High' else 'FF000000')
            cell.alignment = Alignment(horizontal='center', vertical='center')
        elif col_name in ['Brakes Overdue', 'Tyres Overdue'] and cell.value == 'Yes':
            cell.fill = PatternFill('solid', start_color='FFFDE8E8')
            cell.font = Font(bold=True, name='Arial', size=9, color='FFE74C3C')
            cell.alignment = Alignment(horizontal='center', vertical='center')
        elif is_alt and col_name != 'Risk Band':
            cell.fill = PatternFill('solid', start_color=COLOURS['alt_row'])

    # Row height — taller for action column
    ws_sched.row_dimensions[row_idx].height = 30

# Column widths
col_widths = {
    'Registration': 14, 'Make': 12, 'Model': 14, 'Van Type': 12,
    'Driver': 20, 'Branch': 16, 'Vehicle Age (Yrs)': 10, 'Driver Score': 10,
    'Risk Band': 10, 'Risk Score': 9, 'Brakes Overdue': 10, 'Tyres Overdue': 10,
    'Days Since Last Repair': 14, 'Next MOT Date': 13, 'Days Until MOT': 11,
    'Recommended Action': 60
}
for col_idx, col_name in enumerate(cols, start=1):
    ws_sched.column_dimensions[get_column_letter(col_idx)].width = col_widths.get(col_name, 14)

# Auto-filter
ws_sched.auto_filter.ref = ws_sched.dimensions

print(f'Maintenance Schedule sheet built — {len(schedule)} vehicles.')

In [ ]:
# ── SHEET 3: HIGH RISK DETAIL ──────────────────────────────────────────────────
ws_high = wb.create_sheet('High Risk Detail')
ws_high.sheet_view.showGridLines = False
ws_high.freeze_panes = 'A2'

high_risk = schedule[schedule['Risk Band'] == 'High'].copy()

# Title
ws_high.merge_cells(f'A1:{get_column_letter(len(cols))}1')
title = ws_high['A1']
title.value = f'High Risk Vehicles — {GENERATED_DATE}  ({len(high_risk)} vehicles require attention)'
title.font  = Font(bold=True, size=12, name='Arial', color='FFFFFFFF')
title.fill  = PatternFill('solid', start_color=COLOURS['High'])
title.alignment = Alignment(horizontal='center', vertical='center')
ws_high.row_dimensions[1].height = 28

# Header row
for col_idx, col_name in enumerate(cols, start=1):
    header_style(ws_high.cell(row=2, column=col_idx), col_name)
ws_high.row_dimensions[2].height = 28

# Data rows
for row_idx, (_, row_data) in enumerate(high_risk.iterrows(), start=3):
    for col_idx, col_name in enumerate(cols, start=1):
        cell = ws_high.cell(row=row_idx, column=col_idx)
        cell.value = row_data[col_name]
        cell.font  = Font(name='Arial', size=9)
        cell.border = thin_border()
        cell.alignment = Alignment(vertical='center', wrap_text=(col_name == 'Recommended Action'))

        if col_name == 'Risk Band':
            cell.fill = risk_fill('High')
            cell.font = Font(bold=True, name='Arial', size=9, color='FFFFFFFF')
            cell.alignment = Alignment(horizontal='center', vertical='center')
        elif col_name in ['Brakes Overdue', 'Tyres Overdue'] and cell.value == 'Yes':
            cell.fill = PatternFill('solid', start_color='FFFDE8E8')
            cell.font = Font(bold=True, name='Arial', size=9, color='FFE74C3C')
            cell.alignment = Alignment(horizontal='center', vertical='center')
        elif row_idx % 2 == 1:
            cell.fill = PatternFill('solid', start_color=COLOURS['alt_row'])

    ws_high.row_dimensions[row_idx].height = 30

for col_idx, col_name in enumerate(cols, start=1):
    ws_high.column_dimensions[get_column_letter(col_idx)].width = col_widths.get(col_name, 14)

ws_high.auto_filter.ref = f'A2:{get_column_letter(len(cols))}{len(high_risk) + 2}'

print(f'High Risk Detail sheet built — {len(high_risk)} vehicles.')

## 6. Save the Workbook

In [ ]:
output_path = OUTPUTS + 'reports/Fleet_Maintenance_Schedule.xlsx'
wb.save(output_path)

print(f'Workbook saved: {output_path}')
print()
print('Sheets:')
print(f'  1. Summary                — fleet overview and notes')
print(f'  2. Maintenance Schedule   — {len(schedule):,} active vehicles, sorted by risk')
print(f'  3. High Risk Detail       — {len(high_risk):,} high risk vehicles')
print()
print('Risk breakdown:')
for band in ['High', 'Medium', 'Low']:
    count = (schedule['Risk Band'] == band).sum()
    pct   = count / len(schedule) * 100
    print(f'  {band:6s}: {count:>3} vehicles ({pct:.1f}%)')

## 7. Final Project Summary

In [ ]:
high_count   = (schedule['Risk Band'] == 'High').sum()
medium_count = (schedule['Risk Band'] == 'Medium').sum()
low_count    = (schedule['Risk Band'] == 'Low').sum()
brakes_flag  = (schedule.get('Brakes Overdue', pd.Series()) == 'Yes').sum()
tyres_flag   = (schedule.get('Tyres Overdue', pd.Series()) == 'Yes').sum()

print(f"""
========================================================
 FLEET MAINTENANCE RISK REPORT — FINAL SUMMARY
 Generated: {GENERATED_DATE}
========================================================

 Total active vehicles:    {len(schedule):>3}

 RISK BANDS:
   High Risk:              {high_count:>3} vehicles ({high_count/len(schedule)*100:.1f}%)
   Medium Risk:            {medium_count:>3} vehicles ({medium_count/len(schedule)*100:.1f}%)
   Low Risk:               {low_count:>3} vehicles ({low_count/len(schedule)*100:.1f}%)

 SAFETY FLAGS:
   Brakes overdue:         {brakes_flag:>3} vehicles
   Tyres overdue:          {tyres_flag:>3} vehicles

 PROJECT NOTEBOOKS:
   01 — Data Exploration
   02 — Data Cleaning
   03 — Feature Engineering
   04 — Survival Analysis & Rule-Based Flags
   05 — Risk Classification (XGBoost)
   06 — Maintenance Schedule Output

 OUTPUT FILE:
   outputs/reports/Fleet_Maintenance_Schedule.xlsx

 FUTURE IMPROVEMENTS:
   - Incorporate driver change history timeline
   - True odometer readings for mileage-based survival
   - Expand brakes/tyres to survival model as data grows
   - Automate monthly refresh pipeline
   - MOT failure prediction using pre-MOT repair patterns
========================================================
""")